# Теория DPD-кампании −50 dB: полный учебник

Этот notebook — самодостаточный учебник по всей теории и математике,
на которых построена кампания улучшения DPD в репозитории
theJorDea/DPD. Все код-ячейки используют **только синтетические
сигналы** и исполняются на любой машине за секунды; приватные capture
не вшиты.

**Содержание**

1. Физика проблемы: нелинейность и память усилителя
2. Комплексные сигналы, огибающая, NMSE
3. Каскад DPD и почему предыскажение растит пики
4. Модель DPD: сплайновая память, узлы, ветки, фазовая эквивариантность
5. Обучение по записям: наименьшие квадраты и ILA
6. Модели-«приборы» (суррогаты): GMP и фиделити
7. Прямое обучение Гаусса–Ньютона: якобиан конечными разностями
8. Главная ловушка: подгонка под суррогат (surrogate exploitation)
9. Защитный протокол: worst-case, консенсус, разнесённые блоки
10. Программа ёмкости: почему больше веток = лучше
11. Fixed-point: деплой на 16-битной арифметике
12. Информационный потолок: почему −50 требует новых данных
13. Итоги кампании: лестница результатов
14. Словарь терминов

## 1. Физика проблемы: нелинейность и память усилителя

Усилитель мощности (PA) должен делать `y = g·x` — просто умножать вход
на усиление `g`. Реальный PA делает две плохие вещи.

**Нелинейность.** Крупные отсчёты усиливаются слабее, чем нужно
(компрессия), и порождают гармоники. В комплексной форме это удобно
описывать полиномом от амплитуды:

$$y[n] = \sum_k a_k\, x[n]\, |x[n]|^{2k}$$

Множитель `|x|²` = `I² + Q²` — вещественная амплитудная функция; её
степени дают «полиномиальную кривизну», а множитель `x` сохраняет фазу.

**Память.** PA содержит фильтры, транзисторные ёмкости и тепловые
процессы: выход зависит не только от текущего, но и от предыдущих
отсчётов (десятки отсчётов назад). Поэтому честная модель — не функция,
а **оператор с памятью**.

Последствия нелинейности в частотной области — **спектральный рост**
(spectral regrowth): энергия выплёскивается в соседние радиоканалы и
мешает другим абонентам. Стандарты это жёстко нормируют.

### Демонстрация: компрессия и спектральный рост

Ниже мы строим простейший полиномиальный PA (без памяти), подаём
многоканальный OFPC-подобный сигнал и смотрим: (а) как сжимается
амплитудная характеристика AM/AM, (б) как растёт спектральная маска.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
N = 65536
# Многоканальный сигнал: сумма узкополосных подканалов со случайными фазами
freqs = np.array([-3, -2, -1, 1, 2, 3]) * 0.02
x = sum(np.exp(2j * np.pi * f * np.arange(N) + rng.uniform(0, 2*np.pi)) for f in freqs)
x *= 0.35 / np.max(np.abs(x))

# Полиномиальный PA: компрессия + гармонические члены
def pa_poly(u):
    r2 = np.abs(u)**2
    return u * (1.0 - 0.28 * r2 + 0.10 * r2**2 - 0.02 * r2**3)

y = pa_poly(x)
gain = np.vdot(x, y) / np.vdot(x, x)   # комплексный LS-гейн: best y ≈ g·x
err = y - gain * x

print(f"комплексный гейн: {gain:.4f}")
print(f"NMSE(x)      = {10*np.log10(np.mean(np.abs(x - gain*x)**2) / np.mean(np.abs(gain*x)**2)):.2f} dB")
print(f"NMSE выхода  = {10*np.log10(np.mean(np.abs(err)**2) / np.mean(np.abs(gain*x)**2)):.2f} dB  <- искажение PA")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
u = np.abs(x); a = np.abs(y) / np.abs(gain)
ax[0].plot(u[::37], a[::37], '.', ms=1, alpha=0.15)
lim = [0, np.max(u)*1.05]
ax[0].plot(lim, lim, 'k--', lw=1)
ax[0].set_title('AM/AM: компрессия'); ax[0].set_xlabel('|x|'); ax[0].set_ylabel('|y/g|')

w = np.fft.fftshift(np.fft.fftfreq(N))
X = 20*np.log10(np.fft.fftshift(np.abs(np.fft.fft(x))) / N + 1e-12)
Y = 20*np.log10(np.fft.fftshift(np.abs(np.fft.fft(err))) / N + 1e-12)
ax[1].plot(w, X, lw=0.6, label='вход x')
ax[1].plot(w, Y, lw=0.6, alpha=0.8, label='искажение y - g·x')
ax[1].set_ylim(-120, 0); ax[1].set_title('спектральный рост'); ax[1].legend()
ax[1].set_xlabel('норм. частота'); plt.tight_layout(); plt.show()

## 2. NMSE: метрика, в которой живёт вся кампания

$$\mathrm{NMSE} = \frac{\sum_n |y[n] - g\,x[n]|^2}{\sum_n |g\,x[n]|^2},
\qquad \mathrm{dB} = 10\log_{10}(\mathrm{NMSE})$$

Это **доля мощности сигнала, испорченная ошибкой**. Децибелы —
логарифмы по основанию 10: каждые −3 dB ошибка **вдвое** меньше по
мощности; каждые −10 dB — в 10 раз меньше.

| NMSE | доля ошибки | смысл |
|---|---|---|
| −20 dB | 1 % | очень плохо |
| −28.3 dB | 0.15 % | старт кампании (baseline) |
| −31.9 dB | 0.064 % | наш финал DPA |
| −40 dB | 0.01 % | приличная линеаризация |
| −50 dB | 0.001 % | цель Huawei |

Внимание к гейну: NMSE считается **после** подгонки комплексного гейна
`g` (наименьшие квадраты), иначе константное усиление портит метрику.
Также фиксируется **warmup** — первые `W` отсчётов отбрасываются, пока
память моделей заполнена (история = нули).

Тонкость, которая стоила нам пересмотра всех старых чисел: **NMSE
каскада зависит от того, через какую модель PA он измерен**.
Исторические заголовки репозитория (−30.5/−32.4) были сняты через
слабую MP-модель; через сильный GMP-суррогат тот же DPD даёт −28.3.

## 3. Каскад DPD и почему предыскажение растит пики

Каскад, который мы оптимизируем и меряем:

```text
x (желаемый) → DPD: u = D(x) → PA: y = P(u) → хотим y ≈ g·x
```

DPD обязан **предрастягивать** сигнал: раз PA большие отсчёты сжимает,
DPD их должен заранее растянуть. Отсюда два практических следствия:

1. **PAPR драйва выше PAPR входа** — это нормально и неизбежно (в
   кампании: рост пика до +15 %, в экспериментах 1.17–1.27 против 1.0).
2. **Опора (support guard):** если пик драйва вылезает за максимальную
   амплитуду, которую модель PA видела при обучении, модель начинает
   *экстраполировать* и врать. Поэтому любой кандидат проверяется на
   условие `max|u| ≤ max|x|·(1+headroom)`.

## 4. Модель DPD: сплайновая память

Выход DPD — сумма **веток**:

$$u[n] = \sum_{\text{ветки}} x[n-m]\cdot C_{\text{ветка}}\!\left(|x[n-d]|\right)$$

- `x[n−m]` — отсчёт сигнала с задержкой `m` (**память** по сигналу);
- `|x[n−d]|` — огибающая (мгновенная амплитуда) с задержкой `d`
  (нелинейность может зависеть и от «прошлой» амплитуды);
- `C(·)` — нелинейная кривая, **сплайн**: ломаная, заданная значениями
  в **узлах** (knots); между соседними узлами — линейная интерполяция.

**Коэффициенты модели = значения сплайна в узлах** (комплексные: они
поворачивают фазу и меняют амплитуду). Baseline: 3 ветки × 24 узла =
72 комплексных коэффициента. Наш финал: 13 × 24 = 312.

**Почему узлы по квантилям.** Если ставить узлы равномерно по
амплитуде, большая часть разрешения уйдёт в область, где сигнал бывает
редко. Расстановка по квантилям амплитуды (`|x|`) даёт везде одинаковую
«плотность узлов» — в экспериментах выигрывает у равномерной на 4–20 dB.

**Фазовая эквивариантность.** Поворот входа на 90°, `x → j·x`:

- огибающая: `|j·x|² = (−Q)² + I² = I² + Q² = |x|²` — не изменилась
  (сумма вещественных квадратов коммутативна);
- значит `C(|j·x|) = C(|x|)`;
- `x[n−m]·C` → `(j·x[n−m])·C = j·(x[n−m]·C)` — выход каждой ветки
  поворачивается ровно на 90°, и сумма тоже.

Модель не привносит асимметрии I/Q. (Нюанс: float-комплексное умножение
не бит-точно под поворотом на уровне последнего бита — это свойство
IEEE-арифметики; в целочисленном ядре репозитория перестановки точны.)

In [ ]:
# Сплайн: узлы и линейная интерполяция; квантили vs равномерно
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(3)
amp = np.abs(rng.standard_normal(200_000))          # распределение |x|
grid = np.linspace(0, np.max(amp) * 1.15, 1000)

def spline_lookup(xq, knots, values):
    return np.interp(xq, knots, values)

def true_curve(r):                                   # «настоящая» нелинейность
    return 1 + 0.4*np.tanh(2.2*r) - 0.15*r

knots_uniform = np.linspace(0, np.max(amp) * 1.15, 9)
knots_quant = np.quantile(amp, np.linspace(0, 1, 9))
knots_quant[0], knots_quant[-1] = 0.0, np.max(amp) * 1.15

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(grid, true_curve(grid), 'k-', label='истинная кривая')
ax[0].plot(grid, spline_lookup(grid, knots_uniform, true_curve(knots_uniform)), 'r--', label='равномерные узлы')
ax[0].plot(grid, spline_lookup(grid, knots_quant, true_curve(knots_quant)), 'b-.', label='квантильные узлы')
for k in knots_uniform: ax[0].axvline(k, color='r', alpha=0.25, lw=0.5)
for k in knots_quant: ax[0].axvline(k, color='b', alpha=0.25, lw=0.5)
ax[0].set_title('сплайн по узлам'); ax[0].set_xlabel('амплитуда'); ax[0].legend()

ax[1].hist(amp, bins=120, density=True, color='grey', alpha=0.6)
for k in knots_uniform: ax[1].axvline(k, color='r', lw=1)
for k in knots_quant: ax[1].axvline(k, color='b', lw=1)
ax[1].set_title('плотность |x|: красные — равномерно, синие — квантили')
plt.tight_layout(); plt.show()

print("Квантильные узлы следуют за плотностью сигнала: разрешение тратится там, где сигнал живёт.")

## 5. Обучение по записям: наименьшие квадраты и ILA

Пусть модель линейна по параметрам: `u[n] = Σ_j θ_j · φ_j[n]`, где
`φ_j[n]` — известные «словарные функции» (для наших веток:
`x[n−m]·интерполятор(|x[n−d]|)`), а `θ_j` — коэффициенты. Тогда задача
«подобрать θ» — **линейные наименьшие квадраты (LS)**:

$$\min_\theta \| \Phi\theta - t \|_2^2, \qquad
\theta = (\Phi^H\Phi + \lambda I)^{-1}\Phi^H t$$

(`Φ` — матрица словаря: строка = отсчёт, столбец = словарная функция;
`λ` — ridge-регуляризация против вырожденности.)

**Проблема:** DPD надо обучить так, чтобы `P(D(x)) ≈ g·x`, но P входит
внутрь. Трюк репозитория — **ILA (Indirect Learning Architecture)**:
обучить **обратную** модель G на записях: `G(y/g) ≈ x/g`. Если G —
хорошая аппроксимация P⁻¹, то G и есть искомый DPD. Пары берём из
измерений: вход G = измеренный выход усилителя, цель = измеренный вход.
Несколько итераций («заморозили baseline»).

Гейн `g` — тоже LS: `g = ⟨x, y⟩ / ⟨x, x⟩`.

## 6. Суррогаты («приборы»): GMP и фиделити

Чтобы **оценить** кандидата DPD, нужно прогнать предыскажённый сигнал
через PA. Физического PA нет — есть его модель, обученная на записях
«вперёд»: `P_модель: x → y`. В репозитории — GMP (generalized memory
polynomial):

$$\hat y[n] = \sum_{m,d,k} c_{m,d,k}\; x[n-m]\,\big(|x[n-d]|^2\big)^{k}$$

Это тоже линейно по коэффициентам `c` → снова LS. Точность модели
(сравнение `ŷ` с измеренным `y` на validation) — её **фиделити**:

- DPA: **−35.4 dB**; APA: **−38.5 dB** (и вторые, независимо
  построенные, созвучны).

**Ключевая теорема кампании.** Каскад меряется так: `u = D(x)`,
`ŷ = P_модель(u)`, NMSE(ŷ, g·x). Модель P содержит собственную
неустранимую ошибку (шум записей, на которых её учили, + неполноту
архитектуры). Эта ошибка **не зависит от входа**, значит никакой DPD её
не скомпенсирует. Поэтому:

> NMSE каскада ≤ фиделити суррогата (асимптотически).
> **Фиделити прибора = потолок видимости.** Идеальный DPD покажет ≈ −35,
> не −50.

Проверка того, что наши числа не артефакт: два **независимых** суррогата
(разные архитектуры/обучения) дают согласные оценки, а их взаимное
несогласие (−35 dB) — на уровне их собственной фиделити.

## 7. Прямое обучение Гаусса–Ньютона

ILA груб: он не видит настоящий критерий «минимизируй ошибку каскада».
Прямая постановка:

$$\min_\theta \; \big\| g\,x - P(D_\theta(x)) \big\|_2^2$$

Гаусс–Ньютон линеаризует по параметрам θ:

$$P(D_{\theta+\Delta}(x)) \approx P(D_\theta(x)) + J\Delta,
\qquad J_{n,i} = \frac{\partial\, P(D_\theta(x))_n}{\partial \theta_i}$$

**Якобиан конечными разностями** (без аналитических формул):
пошевелить один коэффициент `θ_i` на ε → столбец
`J[:, i] ≈ (P(D_{θ+εe_i}(x)) − P(D_θ(x)))/ε` (с вычетом warmup). При
312 коэффициентах это 313 прогонов модели PA на fit-блоке — CPU
справляется за секунды.

Далее **демпфированный шаг**: решаем

$$\min_\Delta \| J\Delta - r \|_2^2 + \lambda\|\Delta\|_2^2,
\qquad r = g\,x - P(D_\theta(x))$$

с перебором λ (ridge) и длины шага (0.0625…1.0), выбирая кандидата по
критерию (см. § 9). Повторяем на разных непересекающихся парах блоков
train («ротации»).

**Joint stacked objective.** Вместо шага по одному первичному
оценщику — стек остатков и якобианов **обоих** суррогатов:

$$\tilde J = \begin{bmatrix} J_A \\ J_B \end{bmatrix}, \qquad
\tilde r = \begin{bmatrix} r_A \\ r_B \end{bmatrix}$$

— один LS, общие коэффициенты, оба прибора сразу. Ранжирование
кандидатов остаётся worst-case. На DPA это дало +0.38 dB, на APA
оказалось хуже — селекция семейств честно отклонила.

## 8. Главная ловушка: подгонка под суррогат

Если оптимизировать DPD **под один суррогат A**, алгоритм начинает
компенсировать **личные ошибки A** (шум его обучения, неполноту
архитектуры), а не реальную нелинейность PA. Метафора: подгонка под
зерно фотографии — копия идеально совпадает с зерном, но не с камнем.

Экспериментальное доказательство (раунд 1 кампании): словарный кандидат,
обученный под A, показал **+0.78 dB** через A и **−1.3 dB** через
независимый B. Такой «улучшайзер» бесполезен в реальности.

Причина принципиальна: у A и B общие обучающие данные, но разные
архитектуры/шумы инициализации → их ошибки **частично коррелированы, но
не совпадают**. Оптимизатор охотно цепляется за несовпадающую часть.

In [ ]:
# Мини-демо surrogate exploitation: два DPD, обученные под разные суррогаты
import numpy as np

rng = np.random.default_rng(11)
N = 30000
x = (rng.standard_normal(N) + 1j*rng.standard_normal(N)) * 0.4
WARM = 8

def true_pa(u):                       # «физический» усилитель
    r2 = np.abs(u)**2
    return u*(1 - 0.30*r2 + 0.08*r2**2)

def surrogate(u, flavour):            # два суррогата одного объекта
    r2 = np.abs(u)**2
    if flavour == 'A':                # у A — лишний «личный» член (свои ошибки)
        return true_pa(u) + 0.04*np.roll(u, 1)*np.abs(np.roll(u, 1))**2
    return u*(1 - 0.29*r2 + 0.07*r2**2)   # B: чуть иные коэффициенты

def design(v):                        # словарь инверсии: физика + личный roll-член
    r2 = np.abs(v)**2
    cols = [v, v*r2, v*r2**2,
            np.roll(v, 1)*np.roll(r2, 1), np.roll(v, 3)*np.roll(r2, 3)]
    return np.stack(cols, axis=1)

g = np.vdot(x, true_pa(x)) / np.vdot(x, x)
y_A, y_B = surrogate(x, 'A'), surrogate(x, 'B')

def fit_inverse(v_in, target):        # LS: G(v_in) ≈ target
    Phi = design(v_in)
    theta, *_ = np.linalg.lstsq(Phi[WARM:], target[WARM:], rcond=None)
    return theta

theta_A = fit_inverse(y_A / g, x)     # ILA-инверсия, обученная под A (y/g → x)
theta_B = fit_inverse(y_B / g, x)     # ILA-инверсия, обученная под B

def nmse_through(u, forward):         # каскад: forward(G(x)) vs g·x
    y = forward(u)
    return 10*np.log10(np.mean(np.abs(y[WARM:] - g*x[WARM:])**2)
                       / np.mean(np.abs(g*x[WARM:])**2))

def nmse_real(θ):
    u = design(x) @ θ
    return nmse_through(u, true_pa)

def nmse_sur(θ, flavour):
    u = design(x) @ θ
    return nmse_through(u, lambda v: surrogate(v, flavour))

print("Каскадный NMSE, dB (ниже = лучше):")
print(f"  без DPD, через истину:        {nmse_through(x, true_pa):7.2f}")
print(f"  DPD_под_A: через A {nmse_sur(theta_A,'A'):7.2f} | через B {nmse_sur(theta_A,'B'):7.2f} | через истину {nmse_real(theta_A):7.2f}")
print(f"  DPD_под_B: через A {nmse_sur(theta_B,'A'):7.2f} | через B {nmse_sur(theta_B,'B'):7.2f} | через истину {nmse_real(theta_B):7.2f}")
print()
print("Ловушка: глядя ТОЛЬКО через суррогат A, мы выбрали бы DPD_под_A;")
print("но на истинном PA он проигрывает DPD_под_B. «Улучшение» через A — это")
print("компенсация личной ошибки A, которой у реального усилителя нет.")
print("В полной кампании тот же эффект: +0.78 dB через A, -1.3 dB через B.")

## 9. Защитный протокол: три барьера

1. **Worst-case ранжирование.** Кандидат принимается, только если
   улучшает **оба** суррогата; балл = `max(NMSE_A, NMSE_B)`.
2. **Консенсус-обучение.** Для словарных кандидатов целевой вектор —
   **средний остаток** двух суррогатов:
   `t = ( (g·x − P_A(D(x))) + (g·x − P_B(D(x))) ) / 2`.
   Математически это один LS с общими коэффициентами по двум системам —
   несовпадающие ошибки приборов усредняются, а не эксплуатируются.
   Для Гаусса–Ньютона аналог — joint stacked objective (§ 7).
3. **Разнесённые блоки.** `fit` (обучение) / `advisor` (промежуточный
   отбор) / `selection` (финальный отбор) — три непересекающихся куска
   train. Selection никогда не пересекается с fit. Validation — только
   для отчёта. Тест-сплит репозитория не открывался вовсе.

Ещё один барьер — **опора**: `max|u| ≤ max|x|·(1+headroom)`, чтобы
кандидаты не «решали» задачу экстраполяцией модели за диапазон.

## 10. Программа ёмкости: главный прорыв

Никто не доказал, что baseline из 3 веток прав. Мы прогнали **ILA-фит
по измеренным данным** (без суррогатов в самом фите — значит, подгонять
нечего) по семействам веток и узлов, отбирая кандидатов worst-case
протоколом (§ 9), затем GN-полировка.

**Лестница (DPA, worst-case через A и B):**

| Конфигурация | Каскад |
|---|---|
| 3 ветки × 24 узла (baseline) | −28.29 |
| 5 сигнальных веток | −29.83 |
| 7 веток | −30.17 |
| 9 веток | −31.13 |
| 12 веток | −31.43 |
| **13 веток (12 сигнальных + огибающая-2) + joint GN, 4 ротации** | **−31.93** |

APA: 9 веток (5 сигнальных + 4 огибающих) → **−31.37** (насыщение).
Последняя ветка дала +0.08 dB — ёмкостная ось вышла на плато. Важно:
после этого **ни один словарный композит** (GMP-члены поверх сплайна)
не прошёл кросс-гейт — сплайн впитал то, что раньше ловили члены.

## 11. Fixed-point: деплой на 16-битной арифметике

В DSP нет float — есть целые с масштабом. Протокол:

1. выбрать масштаб по пику коэффициентов с guard 1.001;
2. округлить к ближайшему **чётному** (ties-to-even);
3. насыщать при переполнении и **считать насыщения**.

Гейт кампании (16 бит): (а) 0 насыщений; (б) деградация каскадного NMSE
≤ 0.05 dB **через оба суррогата**; (в) потоковая обработка кусками
бит-в-бит равна полной записи; (г) эквивариантность 90° не хуже float.
Все финальные модели — PASS.

Нюанс, который мы вскрыли: float-комплексное умножение **не бит-точно**
под 90°-поворотом даже у замороженного baseline (ошибка 4.4·10⁻¹⁶ —
последний бит IEEE). Бит-точность — свойство целочисленного ядра, где
произведения в int64 и радиус через целочисленный корень делают
перестановки точными.

## 12. Информационный потолок: почему −50 требует новых данных

Соберём цепочку потолков:

```text
[1] записи шумные: собственный шум измерений ≈ −39/−40 dB (аудит репо)
        ↓ наследуется
[2] суррогаты точны до −35.4 (DPA) / −38.5 (APA)
        ↓ теорема § 6
[3] идеальный DPD покажет каскад ≈ −35 максимум
        ↓ наш запас до потолка
[4] мы на −31.9/−31.4 → выжали ≈ 91 % доступного диапазона
```

Каждое звено измерено, не предположено. Чтобы доказать −50, нужно
зерно и прибор на −55…−60 (запас ~10 dB по протоколу репозитория).
Алгоритмом это не лечится: шум записи неотличим от нелинейности.

**Путь вперёд** (в порядке значимости):
1. повторный физический захват: VSA с запасом +15 dB, дробная
   синхронизация ≥ 1/64 отсчёта, коррекции DC/IQ;
2. разнообразие записей (уровни мощности, полосы) и длина;
3. GPU-хост: нейросетевой суррогат фиделити ≥ −55 (обёртки и конфиги в
   репо: `train_opendpd_neural_surrogate.py`, `neural_pa_evaluator.py`).

## 13. Итоги кампании

| | DPA 200MHz | APA 200MHz |
|---|---|---|
| baseline 3×24 | −28.29 | −28.15 |
| **финал** | **−31.93** (A) / −32.40 (B) | **−31.37** (A) / −31.43 (B) |
| сплайн | 12 сигнальных + envelope-2 × 24 узла | 5 сигнальных + 4 огибающих × 24 узла |
| коэффициенты / MUL | 312 / ≈81 | 216 / 63 |
| validation (A/B) | −31.01 / −31.85 | −31.61 / −31.58 |
| 16-bit fixed-point | PASS | PASS |
| итог кампании | **+3.64 dB** | **+3.22 dB** |

Закрытые направления (все с кросс-оценщикным контролем): ёмкость
(насыщение), стратегии узлов (quantile), joint GN (датасет-зависим),
ротации GN (+0.04), композиты поверх финалов (HOLD), широкие словари
1440 членов (HOLD), ILC (негатив −9 dB), нейросудья на CPU (−28.07 —
мало), SPH-судья (без запаса).

**Позитивный протокол кампании** (можно переиспользовать): физический
объект отсутствует → строим ≥ 2 независимых суррогата → любые
улучшения обязаны держаться на **обоих** → отбор только на train-блоках
→ потолок честно считается от фиделити приборов.

## 14. Словарь терминов

| Термин | Смысл |
|---|---|
| PA | Power Amplifier — усилитель мощности |
| DPD | Digital Pre-Distortion — цифровой предыскажатель («анти-очки» PA) |
| NMSE (dB) | доля испорченной мощности сигнала; отрицательнее = лучше |
| I/Q | квадратурные компоненты комплексного отсчёта |
| огибающая | мгновенная амплитуда `\|x\|` |
| сплайн | кусочно-линейная кривая по узлам |
| узел (knot) | точка излома/опоры сплайна; значения в узлах = коэффициенты |
| ветка | член суммы: задержанный сигнал × кривая от огибающей |
| GMP | generalized memory polynomial — форма модели PA с памятью |
| суррогат / оценщик | обученная модель PA; «прибор», через который меряем каскад |
| фиделити | NMSE суррогата против измерений; потолок каскада |
| ILA | indirect learning: обучение обратной функции G(y) ≈ x |
| GN | Гаусс–Ньютон: итеративная линеаризация + LS |
| якобиан | матрица чувствительности выхода к коэффициентам |
| ridge | L2-регуляризация в LS |
| worst-case | балл = max(NMSE_A, NMSE_B) — улучшай обоих |
| консенсус | обучение на среднем остатке двух суррогатов |
| опора (support) | ограничение пика драйва диапазоном обучения суррогата |
| fixed-point | целочисленная арифметика с фиксированной запятой |
| MUL | вещественных умножений на комплексный отсчёт (цена деплоя) |